## Task 5: Predictive Model or Insight Project
Goal:
    Build a simple predictive model or generate meaningful insights from data.
Examples:
    • Simple prediction using regression
    • Classification of data
    • Insight-based recommendation
Key Skills:
    Basic machine learning concepts, model understanding

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [2]:
plt.rcParams.update({
    "figure.facecolor": "#f5f5f0",
    "axes.facecolor":   "#f5f5f0",
    "axes.edgecolor":   "#4a6741",
    "axes.labelcolor":  "#3a5232",
    "xtick.color":      "#3a5232",
    "ytick.color":      "#3a5232",
    "text.color":       "#3a5232",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})
GREEN  = "#4a6741"
TEAL   = "#5b9b8a"
ORANGE = "#e07b39"
PALETTE = [GREEN, ORANGE, TEAL, "#8fbc8f", "#d4a76a"]
sns.set_palette(PALETTE)

In [3]:
df_raw = pd.read_csv("Titanic-Dataset.csv")
df = df_raw.copy()

In [4]:
# 1. Missing values BEFORE
print(f"\n Missing values BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])


 Missing values BEFORE cleaning:
Age         177
Cabin       687
Embarked      2
dtype: int64


In [5]:
# 2. Drop Cabin (>77% missing — not salvageable)
df.drop(columns=["Cabin"], inplace=True)
print("\nDropped 'Cabin' column (77% missing, not recoverable)")


Dropped 'Cabin' column (77% missing, not recoverable)


In [6]:
# 3. Fill Age with median (robust to outliers)
age_median = df["Age"].median()
df["Age"].fillna(age_median, inplace=True)
print(f"Filled {df_raw['Age'].isnull().sum()} missing 'Age' values with median ({age_median})")

Filled 177 missing 'Age' values with median (28.0)


In [7]:
# 4. Fill Embarked with mode (only 2 missing)
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"].fillna(embarked_mode, inplace=True)
print(f"Filled 2 missing 'Embarked' values with mode ('{embarked_mode}')")

Filled 2 missing 'Embarked' values with mode ('S')


In [8]:
# 5. Remove duplicates
dupes_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {dupes_before}")

Duplicates removed: 0


In [9]:
# 6. Feature engineering — extract Title from Name
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df["Title"] = df["Title"].replace(
    ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"],
    "Rare"
)
df["Title"] = df["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
print("Extracted 'Title' feature from Name column")

Extracted 'Title' feature from Name column


In [10]:
# 7. Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
print("Created 'FamilySize' = SibSp + Parch + 1")

Created 'FamilySize' = SibSp + Parch + 1


In [11]:
# 8. Create IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print("Created 'IsAlone' flag")

Created 'IsAlone' flag


In [12]:
# 9. Encode categoricals
df["Sex_enc"]      = LabelEncoder().fit_transform(df["Sex"])
df["Embarked_enc"] = LabelEncoder().fit_transform(df["Embarked"])
df["Title_enc"]    = LabelEncoder().fit_transform(df["Title"])
print("Label-encoded: Sex, Embarked, Title")

Label-encoded: Sex, Embarked, Title


In [13]:
# 10. Drop columns not needed for modelling
df.drop(columns=["PassengerId","Name","Ticket"], inplace=True)

In [14]:
print(f"\nMissing values AFTER cleaning: {df.isnull().sum().sum()} total")
print(f"Final dataset shape: {df.shape}")
print(f"\nSample cleaned rows:")
print(df.head(3).to_string())


Missing values AFTER cleaning: 0 total
Final dataset shape: (891, 14)

Sample cleaned rows:
   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked Title  FamilySize  IsAlone  Sex_enc  Embarked_enc  Title_enc
0         0       3    male  22.0      1      0   7.2500        S    Mr           2        0        1             2          2
1         1       1  female  38.0      1      0  71.2833        C   Mrs           2        0        0             0          3
2         1       3  female  26.0      0      0   7.9250        S  Miss           1        1        0             2          1


In [15]:
survival_rate = df["Survived"].mean() * 100
print(f"\nOverall Survival Rate: {survival_rate:.1f}%")


Overall Survival Rate: 38.4%


In [16]:
print("\n── Survival by Gender ───────────────────────────────────")
print(df.groupby("Sex")["Survived"].agg(["mean","count","sum"])
       .rename(columns={"mean":"Rate","count":"Total","sum":"Survived"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .to_string())



── Survival by Gender ───────────────────────────────────
        Rate  Total  Survived
Sex                          
female  74.2    314       233
male    18.9    577       109


In [17]:
print("\n── Survival by Passenger Class ──────────────────────────")
print(df.groupby("Pclass")["Survived"].agg(["mean","count","sum"])
       .rename(columns={"mean":"Rate","count":"Total","sum":"Survived"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .to_string())


── Survival by Passenger Class ──────────────────────────
        Rate  Total  Survived
Pclass                       
1       63.0    216       136
2       47.3    184        87
3       24.2    491       119


In [18]:
print("\n── Survival by Title ────────────────────────────────────")
print(df.groupby("Title")["Survived"].agg(["mean","count"])
       .rename(columns={"mean":"Rate","count":"Total"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .sort_values("Rate", ascending=False)
       .to_string())


── Survival by Title ────────────────────────────────────
               Rate  Total
Title                     
the Countess  100.0      1
Mrs            79.4    126
Miss           70.3    185
Master         57.5     40
Rare           31.8     22
Mr             15.7    517


In [19]:
print("\n── Age Statistics by Survival ───────────────────────────")
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

# Survival rate within each age group
print(
    df.groupby("AgeGroup")["Survived"]
      .mean()
      .round(3)
      .to_string()
)


── Age Statistics by Survival ───────────────────────────
AgeGroup
Child          0.580
Teen           0.429
Young Adult    0.353
Adult          0.400
Senior         0.227


In [20]:
print("\n── Fare Statistics by Class ─────────────────────────────")
print(df.groupby("Pclass")["Fare"].describe().round(2).to_string())


── Fare Statistics by Class ─────────────────────────────
        count   mean    std  min    25%    50%   75%     max
Pclass                                                      
1       216.0  84.15  78.38  0.0  30.92  60.29  93.5  512.33
2       184.0  20.66  13.42  0.0  13.00  14.25  26.0   73.50
3       491.0  13.68  11.78  0.0   7.75   8.05  15.5   69.55


In [21]:
print("\n── Outlier Check (IQR method) ───────────────────────────")
for col in ["Age","Fare","FamilySize"]:
    Q1, Q3 = df[col].quantile([0.25,0.75])
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
    print(f"  {col:<14}: {outliers} outliers detected")


── Outlier Check (IQR method) ───────────────────────────
  Age           : 66 outliers detected
  Fare          : 116 outliers detected
  FamilySize    : 91 outliers detected


In [30]:
# ══════════════════════════════════════════════════════════
#  TASK 5 — PREDICTIVE MODEL (XGBOOST + FEATURE ENGINEERING)
# ══════════════════════════════════════════════════════════
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier

print("\n" + "═"*60)
print("  TASK 5 — PREDICTIVE MODEL — XGBOOST + FEATURE ENGINEERING")
print("═"*60)

# ── Advanced Feature Engineering ────────────────────────
def engineer_features(df):
    df = df.copy()

    # Age bands (children/elderly have different survival rates)
    df["AgeBand"] = pd.cut(df["Age"].fillna(df["Age"].median()),
                           bins=[0,12,18,35,60,100],
                           labels=[0,1,2,3,4]).astype(float)

    # Fare bands
    df["FareBand"] = pd.qcut(df["Fare"].fillna(df["Fare"].median()),
                              q=4, labels=[0,1,2,3]).astype(float)

    # Interaction: class × gender (1st class women survived most)
    df["Pclass_Sex"] = df["Pclass"] * df["Sex_enc"]

    # Interaction: age × class
    df["Age_Pclass"] = df["Age"].fillna(df["Age"].median()) * df["Pclass"]

    # Is child (under 12)
    df["IsChild"] = (df["Age"].fillna(99) < 12).astype(int)

    # Is elderly (over 60)
    df["IsElderly"] = (df["Age"].fillna(0) > 60).astype(int)

    # Mother indicator (female, over 18, Parch > 0, title is Mrs)
    df["IsMother"] = (
        (df["Sex_enc"] == 1) &
        (df["Age"].fillna(0) > 18) &
        (df["Parch"] > 0) &
        (df["Title_enc"] == 2)
    ).astype(int)

    return df

df_fe = engineer_features(df)

features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "Sex_enc", "Embarked_enc", "Title_enc",
    # Engineered features
    "AgeBand", "FareBand",
    "Pclass_Sex", "Age_Pclass",
    "IsChild", "IsElderly", "IsMother"
]

X = df_fe[features].fillna(df_fe[features].median())
y = df_fe["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n  Train set : {len(X_train)} rows")
print(f"  Test set  : {len(X_test)} rows")
print(f"  Features  : {len(features)} (original + engineered)")

# ── XGBoost Model ────────────────────────────────────────
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
)
xgb_model.fit(
    X_train, y_train,
    verbose=False
)

# ── Soft-Voting Ensemble ─────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=3,
    max_features="sqrt", random_state=42
)
gb_model = GradientBoostingClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)

ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("rf",  rf_model),
        ("gb",  gb_model),
    ],
    voting="soft",
    weights=[3, 2, 2]          
)
ensemble.fit(X_train, y_train)
y_pred = ensemble.predict(X_test)

# ── Results ──────────────────────────────────────────────
acc = accuracy_score(y_test, y_pred)
print(f"\nEnsemble Accuracy : {acc*100:.2f}%")

# Cross-validation for a reliable estimate
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble, X, y, cv=cv, scoring="accuracy")
print(f"10-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

print("\n── Classification Report ────────────────────────────────")
print(classification_report(y_test, y_pred,
      target_names=["Did Not Survive","Survived"]))

# ── Feature Importance (XGBoost native) ─────────────────
xgb_only = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric="logloss", random_state=42
)
xgb_only.fit(X_train, y_train)
imp_df = pd.DataFrame({
    "Feature":    features,
    "Importance": xgb_only.feature_importances_
}).sort_values("Importance", ascending=False)

print("── Feature Importance (XGBoost) ─────────────────────────")
print(imp_df.to_string(index=False))

# ── Plots ─────────────────────────────────────────────────
fig2, axes = plt.subplots(1, 3, figsize=(21, 6), facecolor="#f5f5f0")
fig2.suptitle("Task 5 — XGBoost Ensemble Results",
              fontsize=14, fontweight="bold", color=GREEN)

# 1) Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Predicted 0","Predicted 1"],
            yticklabels=["Actual 0","Actual 1"],
            ax=axes[0], linewidths=1, linecolor="white",
            annot_kws={"size":14, "weight":"bold"})
axes[0].set_title("Confusion Matrix", fontsize=12, fontweight="bold", color=GREEN)

# 2) Feature Importance (top 15)
top15 = imp_df.head(15)
axes[1].barh(top15["Feature"], top15["Importance"], color=GREEN, edgecolor="white")
axes[1].set_title("Top 15 Feature Importances (XGBoost)",
                  fontsize=12, fontweight="bold", color=GREEN)
axes[1].set_xlabel("Importance Score")
axes[1].invert_yaxis()

# 3) CV Score Distribution
axes[2].bar(range(1, 11), cv_scores * 100, color=GREEN, edgecolor="white")
axes[2].axhline(cv_scores.mean() * 100, color=ORANGE,
                linewidth=2, linestyle="--", label=f"Mean: {cv_scores.mean()*100:.2f}%")
axes[2].set_title("10-Fold Cross-Validation Scores",
                  fontsize=12, fontweight="bold", color=GREEN)
axes[2].set_xlabel("Fold")
axes[2].set_ylabel("Accuracy (%)")
axes[2].set_ylim(70, 100)
axes[2].legend()

plt.tight_layout()
plt.savefig("./task5_model.png",
            dpi=150, bbox_inches="tight", facecolor="#f5f5f0")
plt.close()
print("\nSaved: task5_model.png")

# ── Example Predictions ──────────────────────────────────
print("\n── Example Predictions ──────────────────────────────────")
examples_raw = pd.DataFrame([
    {"Pclass":1,"Age":35,"SibSp":1,"Parch":0,"Fare":80,
     "FamilySize":2,"IsAlone":0,"Sex_enc":1,"Embarked_enc":2,"Title_enc":2,
     "AgeBand":2,"FareBand":3,
     "Pclass_Sex":1,"Age_Pclass":35,"IsChild":0,"IsElderly":0,"IsMother":0},
    {"Pclass":3,"Age":22,"SibSp":0,"Parch":0,"Fare":7.25,
     "FamilySize":1,"IsAlone":1,"Sex_enc":0,"Embarked_enc":2,"Title_enc":1,
     "AgeBand":2,"FareBand":0,
     "Pclass_Sex":0,"Age_Pclass":66,"IsChild":0,"IsElderly":0,"IsMother":0},
    {"Pclass":2,"Age":8,"SibSp":0,"Parch":2,"Fare":26,
     "FamilySize":3,"IsAlone":0,"Sex_enc":1,"Embarked_enc":2,"Title_enc":4,
     "AgeBand":0,"FareBand":2,
     "Pclass_Sex":2,"Age_Pclass":16,"IsChild":1,"IsElderly":0,"IsMother":0},
])
preds  = ensemble.predict(examples_raw)
probas = ensemble.predict_proba(examples_raw)[:,1]
labels = [
    "Mrs (1st class, with spouse)",
    "Mr  (3rd class, alone)",
    "Child (2nd class, with parents)"
]
for label, pred, prob in zip(labels, preds, probas):
    outcome = "SURVIVED" if pred == 1 else "Did Not Survive"
    print(f"  {label:<38}: {outcome}  (prob={prob:.2f})")


════════════════════════════════════════════════════════════
  TASK 5 — PREDICTIVE MODEL — XGBOOST + FEATURE ENGINEERING
════════════════════════════════════════════════════════════

  Train set : 712 rows
  Test set  : 179 rows
  Features  : 17 (original + engineered)

Ensemble Accuracy : 82.12%
10-Fold CV Accuracy: 83.95% ± 1.91%

── Classification Report ────────────────────────────────
                 precision    recall  f1-score   support

Did Not Survive       0.82      0.90      0.86       110
       Survived       0.81      0.70      0.75        69

       accuracy                           0.82       179
      macro avg       0.82      0.80      0.81       179
   weighted avg       0.82      0.82      0.82       179

── Feature Importance (XGBoost) ─────────────────────────
     Feature  Importance
     Sex_enc    0.268877
  Pclass_Sex    0.162980
      Pclass    0.151200
    IsMother    0.043000
  FamilySize    0.041037
       SibSp    0.039474
  Age_Pclass    0.036091
Emb